# 02 - Normalization Experiments

This notebook tests different normalization approaches for hand landmarks,
visualizes before/after results, and helps find optimal parameters.

## Contents
1. Test translation normalization
2. Compare scale normalization methods (bbox vs palm-width)
3. Test rotation normalization
4. Visualize normalization effects
5. Find optimal parameters

In [ ]:
# Common imports
import sys
sys.path.insert(0, '..')

from src.types import *
from src.normalize import *
from src.math_utils import *
from src.landmarks import *
from src.validation import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path

# Local visualization utilities
from notebook_utils import (
    plot_hand_3d,
    plot_hand_2d,
    plot_hand_views,
    plot_comparison,
    plot_comparison_overlay,
    plot_multiple_hands,
    plot_variance_heatmap,
    plot_distance_distribution,
    landmarks_to_arrays,
    set_notebook_style,
    create_sample_landmarks,
)

set_notebook_style()
%matplotlib inline

print("Imports loaded successfully!")

In [ ]:
# Create sample data for testing
sample_landmarks = create_sample_landmarks()

# Create variations of the sample (translated, scaled, rotated)
def create_variations(base_landmarks, n=5):
    """Create variations of landmarks for testing normalization."""
    variations = [base_landmarks]
    
    x, y, z = landmarks_to_arrays(base_landmarks)
    base_coords = np.stack([x, y, z], axis=1)
    
    for i in range(n - 1):
        # Apply random translation
        translation = np.random.uniform(-0.3, 0.3, 3)
        translated = base_coords + translation
        
        # Apply random scale
        scale = np.random.uniform(0.7, 1.3)
        scaled = translated * scale
        
        # Convert back to Point3D
        variation = [Point3D(x=p[0], y=p[1], z=p[2]) for p in scaled]
        variations.append(variation)
    
    return variations

# Generate test variations
test_variations = create_variations(sample_landmarks, n=6)
print(f"Created {len(test_variations)} test variations")

# Visualize variations
fig = plot_multiple_hands(
    test_variations,
    titles=[f'Variation {i+1}' for i in range(len(test_variations))],
    ncols=3
)
plt.show()

## 1. Translation Normalization

Move the wrist to the origin to remove position variation.

In [ ]:
# Test translation normalization
def normalize_translation_simple(landmarks, anchor_idx=WRIST):
    """Simple translation normalization to move anchor to origin."""
    x, y, z = landmarks_to_arrays(landmarks)
    
    # Get anchor position
    anchor = landmarks[anchor_idx]
    
    # Translate to origin
    x_norm = x - anchor.x
    y_norm = y - anchor.y
    z_norm = z - anchor.z
    
    return [Point3D(x=x_norm[i], y=y_norm[i], z=z_norm[i]) for i in range(len(landmarks))]

# Apply to all variations
translated = [normalize_translation_simple(v) for v in test_variations]

# Check wrist positions after normalization
for i, t in enumerate(translated):
    wrist = t[WRIST]
    print(f"Variation {i+1} wrist: ({wrist.x:.6f}, {wrist.y:.6f}, {wrist.z:.6f})")

In [ ]:
# Visualize translation normalization
print("Before translation normalization:")
fig = plot_multiple_hands(
    test_variations[:3],
    titles=[f'Original {i+1}' for i in range(3)],
    ncols=3
)
plt.show()

print("\nAfter translation normalization:")
fig = plot_multiple_hands(
    translated[:3],
    titles=[f'Translated {i+1}' for i in range(3)],
    ncols=3
)
plt.show()

In [ ]:
# Test different anchor points
anchors = {
    'wrist': WRIST,
    'palm_center': None,  # Special case
    'middle_mcp': MIDDLE_FINGER_MCP,
}

# Compare palm center (average of palm landmarks)
def normalize_to_palm_center(landmarks):
    """Normalize to palm center instead of wrist."""
    x, y, z = landmarks_to_arrays(landmarks)
    
    # Palm is wrist + base of each finger
    palm_indices = [WRIST, INDEX_FINGER_MCP, MIDDLE_FINGER_MCP, RING_FINGER_MCP, PINKY_MCP]
    
    cx = np.mean([x[i] for i in palm_indices])
    cy = np.mean([y[i] for i in palm_indices])
    cz = np.mean([z[i] for i in palm_indices])
    
    x_norm = x - cx
    y_norm = y - cy
    z_norm = z - cz
    
    return [Point3D(x=x_norm[i], y=y_norm[i], z=z_norm[i]) for i in range(len(landmarks))]

# Compare approaches
v = test_variations[0]
norm_wrist = normalize_translation_simple(v, WRIST)
norm_palm = normalize_to_palm_center(v)
norm_mcp = normalize_translation_simple(v, MIDDLE_FINGER_MCP)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_hand_2d(norm_wrist, ax=axes[0], title='Anchor: Wrist')
plot_hand_2d(norm_palm, ax=axes[1], title='Anchor: Palm Center')
plot_hand_2d(norm_mcp, ax=axes[2], title='Anchor: Middle MCP')
plt.tight_layout()
plt.show()

print("Recommendation: Wrist anchor is most intuitive and matches MediaPipe convention")

## 2. Scale Normalization

Compare bounding box vs palm width approaches.

In [ ]:
# Method 1: Bounding box normalization
def normalize_scale_bbox(landmarks, target_size=1.0):
    """Scale to fit within unit bounding box."""
    x, y, z = landmarks_to_arrays(landmarks)
    
    # Find bounding box
    x_range = x.max() - x.min()
    y_range = y.max() - y.min()
    z_range = z.max() - z.min()
    
    max_range = max(x_range, y_range, z_range)
    if max_range == 0:
        return landmarks
    
    scale = target_size / max_range
    
    # Center before scaling
    cx, cy, cz = x.mean(), y.mean(), z.mean()
    x_scaled = (x - cx) * scale
    y_scaled = (y - cy) * scale
    z_scaled = (z - cz) * scale
    
    return [Point3D(x=x_scaled[i], y=y_scaled[i], z=z_scaled[i]) for i in range(len(landmarks))]

print("Bounding box normalization defined")

In [ ]:
# Method 2: Palm width normalization
def normalize_scale_palm(landmarks, target_width=1.0):
    """Scale based on palm width (wrist to middle MCP)."""
    x, y, z = landmarks_to_arrays(landmarks)
    
    # Calculate palm width (distance from wrist to middle MCP)
    wrist = landmarks[WRIST]
    middle_mcp = landmarks[MIDDLE_FINGER_MCP]
    
    palm_width = np.sqrt(
        (middle_mcp.x - wrist.x)**2 +
        (middle_mcp.y - wrist.y)**2 +
        (middle_mcp.z - wrist.z)**2
    )
    
    if palm_width == 0:
        return landmarks
    
    scale = target_width / palm_width
    
    # Scale from wrist
    x_scaled = (x - wrist.x) * scale
    y_scaled = (y - wrist.y) * scale
    z_scaled = (z - wrist.z) * scale
    
    return [Point3D(x=x_scaled[i], y=y_scaled[i], z=z_scaled[i]) for i in range(len(landmarks))]

print("Palm width normalization defined")

In [ ]:
# Compare both methods
bbox_normalized = [normalize_scale_bbox(v) for v in test_variations]
palm_normalized = [normalize_scale_palm(v) for v in test_variations]

# Measure resulting sizes
def get_size_stats(landmarks_list):
    sizes = []
    palm_widths = []
    
    for landmarks in landmarks_list:
        x, y, z = landmarks_to_arrays(landmarks)
        
        # Bounding box size
        size = max(x.max() - x.min(), y.max() - y.min(), z.max() - z.min())
        sizes.append(size)
        
        # Palm width
        wrist = landmarks[WRIST]
        mcp = landmarks[MIDDLE_FINGER_MCP]
        pw = np.sqrt((mcp.x - wrist.x)**2 + (mcp.y - wrist.y)**2 + (mcp.z - wrist.z)**2)
        palm_widths.append(pw)
    
    return np.array(sizes), np.array(palm_widths)

orig_sizes, orig_palms = get_size_stats(test_variations)
bbox_sizes, bbox_palms = get_size_stats(bbox_normalized)
palm_sizes, palm_palms = get_size_stats(palm_normalized)

print("Size comparison:")
print(f"\nOriginal:")
print(f"  BBox size: {orig_sizes.mean():.4f} +/- {orig_sizes.std():.4f}")
print(f"  Palm width: {orig_palms.mean():.4f} +/- {orig_palms.std():.4f}")

print(f"\nBBox Normalized:")
print(f"  BBox size: {bbox_sizes.mean():.4f} +/- {bbox_sizes.std():.4f}")
print(f"  Palm width: {bbox_palms.mean():.4f} +/- {bbox_palms.std():.4f}")

print(f"\nPalm Normalized:")
print(f"  BBox size: {palm_sizes.mean():.4f} +/- {palm_sizes.std():.4f}")
print(f"  Palm width: {palm_palms.mean():.4f} +/- {palm_palms.std():.4f}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i in range(3):
    plot_hand_2d(bbox_normalized[i], ax=axes[0, i], title=f'BBox Norm {i+1}')
    plot_hand_2d(palm_normalized[i], ax=axes[1, i], title=f'Palm Norm {i+1}')

plt.tight_layout()
plt.suptitle('Scale Normalization Comparison', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Measure variance reduction
if len(test_variations) >= 3:
    print("Variance comparison (lower is more consistent):")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    plot_variance_heatmap.__code__  # Just to show it exists
    
    # Calculate total variance for each method
    def calc_total_variance(landmarks_list):
        coords = []
        for lm in landmarks_list:
            x, y, z = landmarks_to_arrays(lm)
            coords.append(np.stack([x, y, z], axis=1))
        stacked = np.stack(coords, axis=0)
        return np.mean(np.var(stacked, axis=0))
    
    orig_var = calc_total_variance(test_variations)
    bbox_var = calc_total_variance(bbox_normalized)
    palm_var = calc_total_variance(palm_normalized)
    
    print(f"Original variance: {orig_var:.6f}")
    print(f"BBox normalized variance: {bbox_var:.6f} ({(1 - bbox_var/orig_var)*100:.1f}% reduction)")
    print(f"Palm normalized variance: {palm_var:.6f} ({(1 - palm_var/orig_var)*100:.1f}% reduction)")
    
    # Recommendation
    if bbox_var < palm_var:
        print("\n✓ Recommendation: BBox normalization produces more consistent results")
    else:
        print("\n✓ Recommendation: Palm width normalization produces more consistent results")

## 3. Rotation Normalization

Align the hand to a canonical orientation.

In [ ]:
# Rotation normalization using palm plane
def normalize_rotation_palm(landmarks):
    """Rotate hand so palm plane is aligned with XY plane."""
    x, y, z = landmarks_to_arrays(landmarks)
    coords = np.stack([x, y, z], axis=1)
    
    # Define palm plane using wrist, index MCP, and pinky MCP
    wrist = coords[WRIST]
    index_mcp = coords[INDEX_FINGER_MCP]
    pinky_mcp = coords[PINKY_MCP]
    
    # Calculate palm normal
    v1 = index_mcp - wrist
    v2 = pinky_mcp - wrist
    palm_normal = np.cross(v1, v2)
    palm_normal = palm_normal / (np.linalg.norm(palm_normal) + 1e-8)
    
    # Target normal (pointing up in Z)
    target_normal = np.array([0, 0, 1])
    
    # Calculate rotation to align normals
    # Using Rodrigues' rotation formula
    axis = np.cross(palm_normal, target_normal)
    axis_norm = np.linalg.norm(axis)
    
    if axis_norm < 1e-8:
        # Already aligned or opposite
        if np.dot(palm_normal, target_normal) < 0:
            # Flip
            coords[:, 2] = -coords[:, 2]
    else:
        axis = axis / axis_norm
        angle = np.arccos(np.clip(np.dot(palm_normal, target_normal), -1, 1))
        
        # Rotation matrix from axis-angle
        c = np.cos(angle)
        s = np.sin(angle)
        t = 1 - c
        
        R = np.array([
            [t*axis[0]*axis[0] + c, t*axis[0]*axis[1] - s*axis[2], t*axis[0]*axis[2] + s*axis[1]],
            [t*axis[0]*axis[1] + s*axis[2], t*axis[1]*axis[1] + c, t*axis[1]*axis[2] - s*axis[0]],
            [t*axis[0]*axis[2] - s*axis[1], t*axis[1]*axis[2] + s*axis[0], t*axis[2]*axis[2] + c]
        ])
        
        # Apply rotation
        coords = (R @ coords.T).T
    
    return [Point3D(x=coords[i, 0], y=coords[i, 1], z=coords[i, 2]) for i in range(len(landmarks))]

print("Rotation normalization defined")

In [ ]:
# Create rotated test samples
def rotate_landmarks(landmarks, angle_deg, axis='z'):
    """Apply rotation to landmarks."""
    x, y, z = landmarks_to_arrays(landmarks)
    coords = np.stack([x, y, z], axis=1)
    
    angle = np.radians(angle_deg)
    c, s = np.cos(angle), np.sin(angle)
    
    if axis == 'z':
        R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    elif axis == 'y':
        R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    else:  # x
        R = np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
    
    rotated = (R @ coords.T).T
    return [Point3D(x=rotated[i, 0], y=rotated[i, 1], z=rotated[i, 2]) for i in range(len(landmarks))]

# Create variations with different rotations
rotated_samples = [
    sample_landmarks,
    rotate_landmarks(sample_landmarks, 30, 'z'),
    rotate_landmarks(sample_landmarks, -45, 'z'),
    rotate_landmarks(sample_landmarks, 20, 'y'),
]

print("Before rotation normalization:")
fig = plot_multiple_hands(
    rotated_samples,
    titles=['Original', 'Rot Z +30°', 'Rot Z -45°', 'Rot Y +20°'],
    ncols=4
)
plt.show()

In [ ]:
# Apply rotation normalization
rotation_normalized = [normalize_rotation_palm(s) for s in rotated_samples]

print("After rotation normalization:")
fig = plot_multiple_hands(
    rotation_normalized,
    titles=['Norm 1', 'Norm 2', 'Norm 3', 'Norm 4'],
    ncols=4
)
plt.show()

## 4. Combined Normalization Pipeline

Test the full normalization pipeline with optimal parameters.

In [ ]:
# Full normalization pipeline
def normalize_full(landmarks, target_scale=1.0, use_bbox=True):
    """Apply full normalization pipeline."""
    # 1. Translation (wrist to origin)
    normalized = normalize_translation_simple(landmarks, WRIST)
    
    # 2. Scale normalization
    if use_bbox:
        normalized = normalize_scale_bbox(normalized, target_scale)
    else:
        normalized = normalize_scale_palm(normalized, target_scale)
    
    # 3. Rotation normalization (optional, can add noise)
    # normalized = normalize_rotation_palm(normalized)
    
    return normalized

# Test on all variations
fully_normalized = [normalize_full(v, target_scale=1.0) for v in test_variations]

print("Fully normalized samples:")
fig = plot_multiple_hands(
    fully_normalized,
    titles=[f'Normalized {i+1}' for i in range(len(fully_normalized))],
    ncols=3
)
plt.show()

In [ ]:
# Compare before and after
for i in range(min(3, len(test_variations))):
    fig = plot_comparison(
        test_variations[i],
        fully_normalized[i],
        title=f'Sample {i+1}: Before vs After Normalization'
    )
    plt.show()

In [ ]:
# Overlay all normalized samples to see consistency
fig = plot_comparison_overlay(
    fully_normalized[0],
    fully_normalized[1],
    labels=('Sample 1', 'Sample 2'),
    title='Normalized Samples Overlaid'
)
plt.show()

## 5. Find Optimal Parameters

In [ ]:
# Test different target scales
target_scales = [0.5, 1.0, 2.0]

for scale in target_scales:
    normalized = normalize_full(sample_landmarks, target_scale=scale)
    x, y, z = landmarks_to_arrays(normalized)
    
    print(f"Target scale: {scale}")
    print(f"  Actual X range: [{x.min():.3f}, {x.max():.3f}]")
    print(f"  Actual Y range: [{y.min():.3f}, {y.max():.3f}]")
    print(f"  Actual Z range: [{z.min():.3f}, {z.max():.3f}]")
    print()

In [ ]:
# Compare variance with different methods
methods = {
    'Original': test_variations,
    'Translation only': [normalize_translation_simple(v) for v in test_variations],
    'Translation + BBox': [normalize_full(v, use_bbox=True) for v in test_variations],
    'Translation + Palm': [normalize_full(v, use_bbox=False) for v in test_variations],
}

print("Variance comparison by method:")
print("="*60)

variances = []
for name, samples in methods.items():
    var = calc_total_variance(samples)
    variances.append({'method': name, 'variance': var})
    print(f"{name:25s}: {var:.6f}")

var_df = pd.DataFrame(variances)

plt.figure(figsize=(10, 5))
plt.bar(var_df['method'], var_df['variance'])
plt.ylabel('Mean Variance')
plt.title('Variance by Normalization Method')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Test using actual normalize module
print("Testing src.normalize module:")
try:
    # Use the actual normalize functions
    result = normalize_all(sample_landmarks)
    print(f"normalize_all returned {len(result)} landmarks")
    
    fig = plot_comparison(
        sample_landmarks,
        result,
        title='Using src.normalize.normalize_all'
    )
    plt.show()
except Exception as e:
    print(f"Error: {e}")
    print("Check that normalize_all accepts Point3D list directly")

In [ ]:
# Summary of findings
print("="*60)
print("NORMALIZATION RECOMMENDATIONS")
print("="*60)
print("""
1. TRANSLATION: Normalize to wrist at origin
   - Most intuitive anchor point
   - Matches MediaPipe conventions

2. SCALE: Use bounding box normalization
   - More consistent across hand shapes
   - Target scale: 1.0 (unit cube)

3. ROTATION: Optional - use carefully
   - Palm plane alignment can help
   - May introduce noise for non-flat gestures

4. ORDER: Translation -> Scale -> (Optional) Rotation

5. PARAMETERS:
   - target_scale = 1.0
   - anchor_point = WRIST (index 0)
   - scale_method = 'bbox'
""")

## Next Steps

- **03_tolerance_analysis.ipynb** - Analyze variance for tolerance calculation
- **04_angle_calculations.ipynb** - Verify angle calculations
- **05_quality_metrics.ipynb** - Develop quality scoring